# Equity market exposure and macroeconomic associations

**Individual coursework**

This notebook examines Siemens' exposure to the DAX using daily observations from
January 2006 to October 2026, and macroeconomic associations with equity returns
in an eight-country monthly panel from January 2006 to September 2026. The focal
panel comparison concerns lagged nominal effective exchange-rate (NEER)
appreciation in euro-area and other countries. All estimates describe historical
associations rather than causal effects.

Place your authorised workbook at `data/market_macro_data.xlsx` and select
**Restart Kernel and Run All Cells**. Tables and figures are written to the
relative `analysis_outputs/` directory; CSV files retain full numerical precision.
**Source chronology.** Original Bloomberg data were extracted on 12 November 2026;
three supplementary total-return indices were extracted on 16 November 2026.
Benchmark definitions, the two flagged Siemens return observations and DPDF
settings were checked on 16 November 2026. See Sections 2 and 13 for details.

**Reading guide.** Sections 1–3: sources and transformations; 4–5: Siemens–DAX
exposure and stability; 6–8: four main monthly models; 9–11: robustness and
inference sensitivity; 12–13: verification, exports and references.

## 1. Environment and analysis parameters

The following cells define the relative file paths, sample boundaries, country
classifications and inference settings. A shared table formatter provides
consistent display precision without changing the underlying data or exported
values. Package versions are recorded to support reproducibility.

**Software requirements.** Use Python 3 with Jupyter Notebook or JupyterLab,
with `numpy`, `pandas`, `scipy`, `statsmodels`, `matplotlib`, `openpyxl`
and `IPython` installed. Only if dependencies are missing, run this command
in a terminal **before** opening the notebook (not during execution):

```bash
python -m pip install jupyterlab numpy pandas scipy statsmodels matplotlib openpyxl ipython ipykernel
```

The next output records the software versions used for this analysis. No
package installation is performed automatically.

In [ ]:
from pathlib import Path
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import pandas as pd
import scipy
import statsmodels
import statsmodels.formula.api as smf
from IPython.display import display
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan

# Use relative paths so the notebook can run on another computer.
DATA_FILE = Path("data/market_macro_data.xlsx")
OUTPUT_DIR = Path("analysis_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
DAILY_START = pd.Timestamp("2006-01-01")
DAILY_END = pd.Timestamp("2026-10-31")
MONTH_START = pd.Period("2006-01", freq="M")
MONTH_END = pd.Period("2026-09", freq="M")
BREAK_DATE = pd.Timestamp("2018-07-01")
HAC_LAGS = 5  # Prespecified Newey-West lag bandwidth for daily returns.
RECESSION_YEARS = (2009, 2020)  # World Bank (2026); see Section 9.

COUNTRIES = [
    "Germany",
    "Austria",
    "Brazil",
    "Finland",
    "Israel",
    "Italy",
    "Norway",
    "Poland",
]
EURO_COUNTRIES = {"Germany", "Austria", "Finland", "Italy"}

assert DATA_FILE.is_file(), (
    f"Missing {DATA_FILE.name}. Place it next to this notebook. "
    f"Working directory: {Path.cwd()}"
)

In [ ]:
def show_table(frame, caption, formats=None, keep_index=False):
    """Format a displayed table without changing numeric data or CSV
    exports.
    """
    # Display very small p-values as thresholds rather than rounding to 0.0000.
    def _format_pvalue(value):
        if pd.isna(value):
            return "—"
        return "<0.001" if 0 <= value < 0.001 else f"{value:.3f}"

    pvalue_formats = {
        col: _format_pvalue
        for col in frame.columns
        if isinstance(col, str)
        and ("p-value" in col.lower() or "p_value" in col.lower() or col.lower() == "p")
    }
    display_formats = {**pvalue_formats, **(formats or {})}
    table = (
        frame.style.format(display_formats, precision=4, na_rep="—")
        .set_caption(caption)
        .set_table_styles([
            {"selector": "caption", "props": [
                ("caption-side", "top"), ("text-align", "left"),
                ("font-weight", "bold"), ("font-size", "1.03em"),
                ("padding-bottom", "7px"), ("color", "#1F3348")
            ]},
            {"selector": "th", "props": [
                ("background-color", "#EDF2F7"),
                ("font-weight", "600"), ("padding", "6px 9px"),
                ("border-bottom", "1px solid #BAC6D3")
            ]},
            {"selector": "td", "props": [
                ("padding", "5px 9px"),
                ("border-bottom", "1px solid #E2E8F0")
            ]},
            {"selector": "tbody tr:nth-child(even)", "props": [
                ("background-color", "#F8FAFC")
            ]}
        ])
    )
    if not keep_index:
        table = table.hide(axis="index")
    display(table)


TERM_LABELS = {
    "inflation_lag_pct": "Lagged inflation",
    "policy_lag_pct": "Lagged policy rate",
    "neer_lag_appreciation_pct": "Lagged NEER appreciation",
    "neer_lag_appreciation_pct:euro": "NEER × euro",
}
plt.rcParams.update({"font.size": 10, "axes.titlesize": 12, "figure.dpi": 110})

In [ ]:
# Record the exact software versions used to generate reported results.
package_versions = pd.Series(
    {
        "Python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "statsmodels": statsmodels.__version__,
        "matplotlib": matplotlib.__version__,
        "openpyxl": openpyxl.__version__,
    },
    name="Version",
)

print("Input workbook:", DATA_FILE.name)
show_table(
    package_versions.to_frame().rename_axis("Package"),
    "Software versions",
    keep_index=True,
)

## 2. Data sources, cleaning and coverage

The workbook supplies daily Siemens and DAX levels and monthly equity, CPI
inflation, policy-rate and national broad nominal NEER observations for Germany,
Austria, Brazil, Finland, Israel, Italy, Norway and Poland. Calculations use these
source-level worksheets, not the workbook's stored returns or regression
results. The input reader checks dates, numerical entries and ordering; subsequent
checks validate columns, observed index positivity and monthly calendar coverage.

Empty observations remain missing. No interpolation or forward-filling is applied
in this notebook. The coverage table documents the observations available before
return construction, lagging and complete-case exclusions.

In [ ]:
def read_source(workbook, sheet_name):
    """Read row-3 headers; validate dates and numeric values, preserving
    missing data.
    """
    frame = pd.read_excel(workbook, sheet_name=sheet_name, header=2)
    frame = frame.dropna(how="all").copy()

    date_column = frame.columns[0]
    frame[date_column] = pd.to_datetime(frame[date_column], errors="raise")
    assert frame[date_column].notna().all(), f"Missing dates: {sheet_name}"
    assert frame[date_column].is_unique, f"Duplicate dates: {sheet_name}"
    assert frame[
        date_column
    ].is_monotonic_increasing, f"Unsorted dates: {sheet_name}"

    for column in frame.columns[1:]:
        # Empty Bloomberg cells are missing, not zero. Unexpected text is
        # flagged instead of silently converted into another missing value.
        raw = frame[column]
        numeric = pd.to_numeric(raw, errors="coerce")
        nonempty = raw.notna() & raw.astype("string").str.strip().ne("")
        unexpected_text = nonempty & numeric.isna()
        assert not unexpected_text.any(), (
            f"Non-numeric entry in {sheet_name}/{column}: "
            f"rows {frame.index[unexpected_text].tolist()[:5]}"
        )
        frame[column] = numeric

    return frame.reset_index(drop=True)

In [ ]:
source_sheets = {
    "equity": "Monthly_Equity",
    "inflation": "Monthly_Inflation",
    "policy": "Monthly_Policy",
    "neer": "Monthly_NEER",
}

with pd.ExcelFile(DATA_FILE) as workbook:
    daily_levels = read_source(workbook, "Daily_Prices")
    monthly_sources = {
        name: read_source(workbook, sheet_name)
        for name, sheet_name in source_sheets.items()
    }

In [ ]:
assert list(daily_levels.columns) == [
    "Date",
    "Siemens_adjusted_level",
    "DAX_level",
], "The daily source columns have changed."
assert (
    not (daily_levels.iloc[:, 1:] <= 0).any().any()
), "Daily levels must be positive wherever observed."

for name, frame in monthly_sources.items():
    assert list(frame.columns) == [
        "Month",
        *COUNTRIES,
    ], f"Unexpected country columns: {name}"
    months = frame["Month"].dt.to_period("M")
    expected = pd.period_range(months.min(), months.max(), freq="M")
    assert months.is_unique, f"Repeated country-months: {name}"
    assert len(months) == len(
        expected
    ), f"Missing calendar months in {name}; check before lagging."
    if name in {"equity", "neer"}:
        assert (
            not (frame[COUNTRIES] <= 0).any().any()
        ), f"Non-positive index level in {name}"

In [ ]:
# Save a compact provenance/coverage audit alongside the estimates.
coverage_rows = []
for name, frame in {"daily": daily_levels, **monthly_sources}.items():
    observation_dates = frame.iloc[:, 0]
    coverage_rows.append(
        {
            "dataset": name,
            "rows": len(frame),
            "first": str(observation_dates.min().date()),
            "last": str(observation_dates.max().date()),
            "missing_values": int(frame.iloc[:, 1:].isna().sum().sum()),
        }
    )

coverage = pd.DataFrame(coverage_rows)
show_table(
    coverage.rename(
        columns={
            "dataset": "Dataset",
            "rows": "Rows",
            "first": "First observation",
            "last": "Last observation",
            "missing_values": "Missing values",
        }
    ),
    "Source coverage before return and lag construction",
)

**Verified Bloomberg and index-provider conventions (16 November 2026).**
The original Siemens (`SIE GY Equity`) and DAX (`DAX Index`) Excel historical
formulas requested `PX_LAST` with `UseDPDF=N`, `CshAdjNormal=Y`,
`CshAdjAbnormal=Y` and `CapChg=Y`. These are the *requested extraction
settings*, not a claim that every dividend or corporate-action event has been
individually audited. The default terminal DPDF screen is not proof of the
historical formula settings. The standard `DAX Index` is a performance index.

The eight monthly benchmark conventions, checked against Bloomberg's DES
profiles and index-provider documentation, are:

| Country | Bloomberg ticker | Benchmark convention |
|:--|:--|:--|
| Germany | `DAX Index` | Gross total return (performance) |
| Austria | `ATX Index` | Price return |
| Brazil | `IBOV Index` | Gross total return |
| Finland | `HEX25 Index` | Price return |
| Israel | `TA-125 Index` | Gross total return |
| Italy | `FTSEMIB Index` | Price return |
| Norway | `OSEBX Index` | Total return |
| Poland | `WIG Index` | Total return, including subscription rights |

This means **five total-return/performance series and three price-return series**;
changes in these levels are not exactly homogeneous equity-return measures.
Country effects do not remove time-varying differences in dividend yields.
The classifications describe the *underlying index definitions*; they do not
imply that the raw Bloomberg export has been independently rebuilt.

**Flagged Siemens days.** Historical prices displayed in Bloomberg HP were
checked for the return observations on 17 March and 29 October 2008. The
price movements agreed with the calculated returns and both observations
were retained. This check corroborates the changes but does not independently
establish their economic causes or audit every corporate-action adjustment.

**Remaining data limitations.** Austria's November and December 2005 CPI
observations are missing, preventing inclusion of January 2006 in the
complete-case regression panel. CPI dates are reference months, not release
timestamps: one-month lags do not guarantee information was public before the
return month. The four euro-area countries share an ECB policy-rate reference;
the BIS-documented September 2024 instrument change complicates comparability.
The notebook reconstructs analysis from the retained workbook input sheets, not
from a new Bloomberg download.


**Additional harmonised-index sources (16 November 2026).** A separate
robustness analysis uses Bloomberg monthly `PX_LAST` levels for Austria
(`ATXTR Index`), Finland (`OMXH25GI Index`) and Italy (`TFTSEMIB Index`),
recorded in the `TR_*` sheets of the accompanying Excel workbook. These
are the corresponding gross total-return benchmarks. The three original Bloomberg
Excel export formulas each specify `UseDPDF=Y`, rather than explicit cash-
dividend adjustments. On the terminal's **DPDF Corporate Action Settings** screen
inspected on **16 November 2026**, *Historical Settings* showed **Normal Cash
Dividends: off**, **Abnormal Cash Dividends: off**, **Historical Pricing: on**
and **Historical Volume: on**. *Cash Dividend Intraday: on* was a separate
intraday setting, not a monthly `PX_LAST` adjustment. With the photographed
settings, an extra historical cash-dividend adjustment was not requested for
these three exports; this is not a retroactive event-by-event audit or proof
of the exact historical configuration at every earlier date. The original
Siemens/DAX extraction used the separate explicit settings described above. Finland's first
observed level is June 2014, so comparable monthly returns begin July
2014. The original index levels, 2006–2026 panel and four principal
regressions remain unchanged. The source register records all three
original Bloomberg export filenames.


## 3. Daily returns and descriptive evidence

Daily simple percentage returns are calculated from successive aligned price
observations, using December 2005 history to construct the start of the assigned
sample. Rows without a usable stock–market return pair are excluded without
filling prices. The tables report the estimation sample, return distribution and
largest absolute daily moves for source verification.

In [ ]:
# Compute percentage returns on aligned dates without carrying prices forward.
prices = daily_levels.set_index("Date").sort_index()
daily_returns = 100 * prices[
    ["Siemens_adjusted_level", "DAX_level"]
].pct_change(fill_method=None)

daily_returns = daily_returns.rename(
    columns={
        "Siemens_adjusted_level": "siemens_return_pct",
        "DAX_level": "dax_return_pct",
    }
)
daily_sample = (
    daily_returns.loc[DAILY_START:DAILY_END]
    .dropna(subset=["siemens_return_pct", "dax_return_pct"])
    .reset_index()
)

# Define the two independent comparison designs used in Part I.
daily_sample["post_2018"] = (daily_sample["Date"] >= BREAK_DATE).astype(int)
daily_sample["down_market"] = (daily_sample["dax_return_pct"] < 0).astype(int)

assert (
    not daily_sample.empty and daily_sample["Date"].is_unique
), "Invalid daily return index."
assert daily_sample["Date"].between(DAILY_START, DAILY_END).all()

In [ ]:
print(
    f"Daily observations: {len(daily_sample):,} | "
    f"Pre-break: {(daily_sample['post_2018'] == 0).sum():,} | "
    f"Post-break: {(daily_sample['post_2018'] == 1).sum():,}"
)
show_table(
    daily_sample[["siemens_return_pct", "dax_return_pct"]]
    .describe()
    .T.rename(
        index={"siemens_return_pct": "Siemens", "dax_return_pct": "DAX"}
    ),
    "Daily return distribution (%)",
    formats={"count": "{:,.0f}"},
    keep_index=True,
)

# Highlight extreme observations for source/event review, not automatic
# deletion.
extreme_days = daily_sample.assign(
    maximum_absolute_return=daily_sample[
        ["siemens_return_pct", "dax_return_pct"]
    ]
    .abs()
    .max(axis=1)
).nlargest(12, "maximum_absolute_return")

show_table(
    extreme_days[["Date", "siemens_return_pct", "dax_return_pct"]].rename(
        columns={
            "siemens_return_pct": "Siemens (%)",
            "dax_return_pct": "DAX (%)",
        }
    ),
    "Largest absolute daily moves: source verification",
    formats={"Date": "{:%Y-%m-%d}"},
)

## 4. Daily market exposure and model adequacy

The full-sample market model estimates Siemens' contemporaneous association with
the domestic market:

\[
R^S_t = \alpha + \beta R^M_t + \varepsilon_t,
\]

where both returns are expressed in per cent. The coefficient \(\beta\) measures
market exposure. Because the regression uses raw rather than excess returns,
\(\alpha\) is not interpreted as a risk-free-adjusted CAPM or Jensen alpha.
Conventional OLS and Newey–West HAC(5) inference are reported for the same fitted
coefficients.

P-values are displayed to three significant figures. A displayed zero reflects
floating-point underflow rather than an exact zero probability.

### Working assumptions: daily market model

| Assumption | Assessment for this analysis |
|:--|:--|
| **Linear market relationship** | The OLS slope is a constant-beta approximation to the conditional stock–market association over each estimation window. Section 5 tests a specified time break; the rolling beta shows that stability is not guaranteed. |
| **Exogeneity / identification** | A causal interpretation would require suitable exogeneity of DAX returns relative to unobserved Siemens shocks. Contemporaneous market and stock returns do not establish this; beta is interpreted as a descriptive exposure, not a causal effect. |
| **Classical error variance and serial independence** | Breusch–Pagan and Ljung–Box identify departures from these assumptions; serial dependence may also affect the conventional Breusch–Pagan reference p-value. We therefore report HAC inference instead of relying on classical OLS standard errors. |
| **HAC approximation** | Newey–West HAC(5) presumes sufficiently weak/decaying dependence and a defensible bandwidth. Five daily lags approximate a trading week, not a uniquely validated choice. Robust covariance estimation changes uncertainty estimates, not the fitted coefficient or the model's identification. |

These assumptions distinguish why the estimated **historical association** can be reported from why neither unbiased causal identification nor complete robustness to all dependence structures is claimed.

In [ ]:
# Baseline OLS market model and HAC inference (same point estimates).
base_ols = smf.ols(
    "siemens_return_pct ~ dax_return_pct", data=daily_sample
).fit()
base_hac = base_ols.get_robustcov_results(
    cov_type="HAC", maxlags=HAC_LAGS, use_t=False
)

daily_coefficients = pd.DataFrame(
    {
        "coefficient": base_ols.params,
        "conventional_SE": base_ols.bse,
        "HAC5_SE": base_hac.bse,
        "HAC5_p": base_hac.pvalues,
    }
)
show_table(
    daily_coefficients.rename(
        index={"Intercept": "Intercept", "dax_return_pct": "Market beta"},
        columns={
            "coefficient": "Estimate",
            "conventional_SE": "Conventional SE",
            "HAC5_SE": "HAC(5) SE",
            "HAC5_p": "HAC(5) p-value",
        },
    ),
    f"Daily market model: conventional and HAC({HAC_LAGS}) inference",
    formats={
        "Estimate": "{:.5f}",
        "Conventional SE": "{:.5f}",
        "HAC(5) SE": "{:.5f}",
        "HAC(5) p-value": "{:.3g}",
    },
    keep_index=True,
)
print(f"N = {int(base_ols.nobs):,}; R-squared = {base_ols.rsquared:.4f}")

### Residual diagnostics

Breusch–Pagan assesses constant residual variance; Ljung–Box assesses joint serial
correlation through five daily lags, with 20 lags retained as the longer-window
check. Their principal null hypotheses are

\[
H_{0,\mathrm{BP}}:\operatorname{Var}(\varepsilon_t\mid R^M_t)=\sigma^2,
\qquad
H_{0,\mathrm{LB}}:\rho_1=\cdots=\rho_5=0.
\]

Lecture 03 (slides 35 and 41–42) covers the variance diagnostic and Lecture 04
(slide 10) covers Ljung–Box. Lecture 02 (slides 37–38) distinguishes OLS point
estimates from robust covariance estimation. HAC extends the covariance treatment
to serial dependence; it is distinct from the HC/White estimator in those slides.

In [ ]:
# Diagnostic 1: H0 = constant residual variance (Breusch-Pagan).
bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(
    base_ols.resid, base_ols.model.exog
)
# Diagnostic 2: H0 = no serial correlation up to the specified lag.
ljung_box = acorr_ljungbox(base_ols.resid, lags=[5, 20], return_df=True)

diagnostic_evidence = pd.DataFrame(
    [
        {
            "Diagnostic": "Breusch-Pagan LM",
            "Statistic": bp_lm,
            "p-value": bp_lm_p,
        },
        {
            "Diagnostic": "Breusch-Pagan F",
            "Statistic": bp_f,
            "p-value": bp_f_p,
        },
        {
            "Diagnostic": "Ljung-Box (5 lags)",
            "Statistic": ljung_box.loc[5, "lb_stat"],
            "p-value": ljung_box.loc[5, "lb_pvalue"],
        },
        {
            "Diagnostic": "Ljung-Box (20 lags)",
            "Statistic": ljung_box.loc[20, "lb_stat"],
            "p-value": ljung_box.loc[20, "lb_pvalue"],
        },
    ]
)
show_table(
    diagnostic_evidence,
    "Daily residual diagnostics: test statistics and p-values",
    formats={"Statistic": "{:.3f}", "p-value": "{:.3g}"},
)

In [ ]:
# Use the same fitted coefficients for the figure and regression table.
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.scatter(
    daily_sample["dax_return_pct"],
    daily_sample["siemens_return_pct"],
    alpha=0.13,
    s=8,
)
market_grid = np.linspace(
    daily_sample["dax_return_pct"].min(),
    daily_sample["dax_return_pct"].max(),
    250,
)
ax.plot(
    market_grid,
    base_ols.params["Intercept"]
    + base_ols.params["dax_return_pct"] * market_grid,
    linewidth=2,
    color="black",
    label="Fitted market model",
)
ax.set(
    xlabel="DAX daily return (%)",
    ylabel="Siemens daily return (%)",
    title="Daily market exposure: Siemens vs DAX",
)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "part1_scatter.png", dpi=155)
plt.show()
plt.close(fig)

The positive estimated beta indicates substantial exposure to German market
movements. Both diagnostics reject their respective null hypotheses at 5%,
supporting caution with conventional standard errors. HAC inference addresses
heteroscedasticity and serial dependence under the specified bandwidth; it does
not establish exogeneity or a causal interpretation, and diagnostic rejection
alone does not demonstrate bias in beta.

## 5. Changes in market exposure

The pre-/post-July 2018 comparison uses

\[
R^S_t=\alpha+\beta R^M_t+\gamma D_t
       +\delta(D_tR^M_t)+\varepsilon_t,
\]

where \(D_t=1\) from 1 July 2018. Pre-period beta is \(\beta\), post-period beta is
\(\beta+\delta\), and the restrictions are

\[
H_{0,\beta}:\delta=0,
\qquad H_{0,\mathrm{joint}}:\gamma=\delta=0.
\]

These restrictions follow Lecture 03 (slides 26–32), using the specified HAC
covariance rather than the spherical-error covariance of the classical
RSS-based F-test. Separate subperiod estimates are reported alongside the
formal tests.

In [ ]:
# Include a post-June-2018 indicator and a market-return interaction.
# H0(slope): the interaction coefficient equals zero.
# H0(joint): neither intercept nor beta changes across the break.
interaction = smf.ols(
    "siemens_return_pct ~ dax_return_pct * post_2018", data=daily_sample
).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS})

period_rows = []
for post_break, label in [
    (0, "Jan 2006–Jun 2018"),
    (1, "Jul 2018–Oct 2026"),
]:
    subperiod = daily_sample.loc[daily_sample["post_2018"] == post_break]
    subperiod_model = smf.ols(
        "siemens_return_pct ~ dax_return_pct", data=subperiod
    ).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS})
    period_rows.append(
        {
            "period": label,
            "N": int(subperiod_model.nobs),
            "beta": subperiod_model.params["dax_return_pct"],
            "HAC5_SE": subperiod_model.bse["dax_return_pct"],
            "R_squared": subperiod_model.rsquared,
        }
    )

period_comparison = pd.DataFrame(period_rows)

In [ ]:
show_table(
    period_comparison.rename(
        columns={
            "period": "Period",
            "beta": "Beta",
            "HAC5_SE": "HAC SE",
            "R_squared": "R-squared",
        }
    ),
    "Market exposure before and after July 2018",
    formats={"Beta": "{:.5f}", "HAC SE": "{:.5f}"},
)

slope_test = interaction.t_test("dax_return_pct:post_2018 = 0")
joint_test = interaction.wald_test(
    "post_2018 = 0, dax_return_pct:post_2018 = 0",
    scalar=True,
)

stability_tests = pd.DataFrame(
    [
        {"test": "Equal pre/post beta", "p_value": float(slope_test.pvalue)},
        {
            "test": "Joint intercept and beta stability",
            "p_value": float(joint_test.pvalue),
        },
    ]
)
show_table(
    stability_tests.rename(
        columns={"test": "Null hypothesis", "p_value": "p-value"}
    ),
    "Formal stability tests: HAC inference",
    formats={"p-value": "{:.3g}"},
)

### Market-state comparison and rolling beta

The supplementary interaction tests whether beta differs between negative and
non-negative DAX-return days. This is a separate restriction from the calendar
break. The 126-observation rolling-beta figure is descriptive and does not
replace either formal interaction test.

In [ ]:
# Supplementary sign-dependent beta test (distinct from the 2018 break).
market_state_model = smf.ols(
    "siemens_return_pct ~ dax_return_pct * down_market", data=daily_sample
).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS})

print(
    "Down-market beta difference:",
    round(market_state_model.params["dax_return_pct:down_market"], 5),
    "p =",
    round(market_state_model.pvalues["dax_return_pct:down_market"], 6),
)

In [ ]:
# Rolling beta = rolling stock-market covariance / rolling market variance.
daily_series = daily_sample.set_index("Date")
rolling_covariance = (
    daily_series["siemens_return_pct"]
    .rolling(126, min_periods=100)
    .cov(daily_series["dax_return_pct"])
)
rolling_variance = (
    daily_series["dax_return_pct"].rolling(126, min_periods=100).var()
)
rolling_beta = rolling_covariance / rolling_variance

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rolling_beta.index, rolling_beta, linewidth=0.9)
ax.axvline(
    BREAK_DATE,
    color="black",
    linestyle="--",
    linewidth=1,
    label="July 2018 break",
)
ax.set(
    xlabel="Date",
    ylabel="126-day rolling OLS beta",
    title="Siemens market exposure over time",
)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "part1_rolling_beta.png", dpi=155)
plt.show()
plt.close(fig)

The July 2018 interaction indicates a statistically detectable change in beta,
whereas the down-market interaction is not significant at 5%. The calendar
comparison identifies a change in association, not a causal effect of trade
tensions. The rolling estimates describe how exposure varies within the sample.

## 6. Monthly panel construction

The panel is aligned by calendar month and country before changes or lags are
calculated. Equity returns are simple percentage changes; \(q_{it}\) denotes
100 times the natural-log change in the country's broad nominal NEER, with
positive values indicating appreciation. National NEER series are retained for
euro-area members as well as other countries.

The regressors are the previous reference month's year-on-year CPI
inflation \(\pi_{i,t-1}\), nominal policy rate \(r_{i,t-1}\), and NEER appreciation
\(q_{i,t-1}\). Inflation and policy rates are recorded in per cent. Returns and
lags are constructed before sample restrictions; the models then use a common
complete-case sample. Coverage and country summaries below retain the evidence
on exclusions and comparability.

In [ ]:
def to_long(wide, value_name):
    """Convert one wide monthly country dataset to country-month
    observations.
    """
    with_month = wide.assign(month=wide["Month"].dt.to_period("M"))
    return with_month.melt(
        id_vars="month",
        value_vars=COUNTRIES,
        var_name="country",
        value_name=value_name,
    )


panel = to_long(monthly_sources["equity"], "equity_level")
for source_name, value_name in [
    ("inflation", "inflation_yoy_pct"),
    ("policy", "policy_rate_pct"),
    ("neer", "neer_index"),
]:
    panel = panel.merge(
        to_long(monthly_sources[source_name], value_name),
        on=["country", "month"],
        how="outer",
        validate="one_to_one",
    )

panel = panel.sort_values(["country", "month"]).reset_index(drop=True)
panel["euro"] = panel["country"].isin(EURO_COUNTRIES).astype(int)

In [ ]:
# Construct returns within country, preventing cross-country leakage.
grouped = panel.groupby("country", sort=False)
panel["equity_return_pct"] = 100 * (
    panel["equity_level"] / grouped["equity_level"].shift(1) - 1
)
panel["neer_appreciation_pct"] = 100 * np.log(
    panel["neer_index"] / grouped["neer_index"].shift(1)
)

# Explanatory variables refer to the previous CALENDAR month, not the
# preceding non-missing row after sample restrictions.
panel["inflation_lag_pct"] = grouped["inflation_yoy_pct"].shift(1)
panel["policy_lag_pct"] = grouped["policy_rate_pct"].shift(1)
panel["neer_lag_appreciation_pct"] = grouped["neer_appreciation_pct"].shift(1)

# Apply the assigned sample boundaries AFTER computing changes and lags.
panel = panel.loc[panel["month"].between(MONTH_START, MONTH_END)].copy()
panel["month_id"] = panel["month"].astype(str)
panel["year"] = panel["month"].dt.year

macro_terms = [
    "inflation_lag_pct",
    "policy_lag_pct",
    "neer_lag_appreciation_pct",
]
regression_columns = ["equity_return_pct", *macro_terms]
monthly_sample = panel.dropna(subset=regression_columns).copy()

In [ ]:
n_months = len(pd.period_range(MONTH_START, MONTH_END, freq="M"))
assert (
    panel.groupby("country").size().eq(n_months).all()
), "Country-month panel coverage differs from the assigned sample."
assert not monthly_sample[["country", "month"]].duplicated().any()
assert not monthly_sample.empty
assert monthly_sample["country"].nunique() == len(COUNTRIES)

# The euro-area policy-rate series should be identical across the four members.
euro_policy_unique = (
    panel.loc[panel["euro"] == 1].groupby("month")["policy_rate_pct"].nunique()
)
assert (
    euro_policy_unique.max() == 1
), "Unexpected divergence in ECB policy data."

In [ ]:
print(
    f"Potential country-months: {len(panel):,} | "
    f"Complete-case regression rows: {len(monthly_sample):,}"
)
availability = panel.groupby("country")[regression_columns].count()
show_table(
    availability.rename(
        columns={"equity_return_pct": "Equity return", **TERM_LABELS}
    ),
    "Non-missing observations by country and regression variable",
    keep_index=True,
)

In [ ]:
country_summary = monthly_sample.groupby("country").agg(
    N=("equity_return_pct", "size"),
    mean_equity_return=("equity_return_pct", "mean"),
    sd_equity_return=("equity_return_pct", "std"),
    mean_lagged_CPI=("inflation_lag_pct", "mean"),
    mean_lagged_policy=("policy_lag_pct", "mean"),
    mean_lagged_NEER_change=("neer_lag_appreciation_pct", "mean"),
)
show_table(
    country_summary.rename(
        columns={
            "mean_equity_return": "Mean return (%)",
            "sd_equity_return": "Return SD (%)",
            "mean_lagged_CPI": "Mean inflation (%)",
            "mean_lagged_policy": "Mean policy rate (%)",
            "mean_lagged_NEER_change": "Mean NEER change (%)",
        }
    ),
    "Monthly complete-case sample: country summary",
    keep_index=True,
)

## 7. Monthly models and explanatory power

The four specifications progressively account for stable country characteristics
and common monthly shocks. Each includes all three lagged macroeconomic
variables on the same complete-case sample.

| Model | Intercept structure and additional terms |
|---|---|
| M1 | Common intercept; three lagged macro variables |
| M2 | Country fixed effects; three lagged macro variables |
| M3 | Country and calendar-month fixed effects; three lagged macro variables |
| M4 | M3 plus lagged NEER appreciation interacted with the euro-area indicator |

The focal specification is

\[
R_{it}=\alpha_i+\tau_t+\beta_\pi\pi_{i,t-1}
       +\beta_r r_{i,t-1}+\beta_q q_{i,t-1}
       +\delta(E_iq_{i,t-1})+u_{it},
\]

where \(E_i\) identifies Germany, Austria, Finland and Italy. Its main effect is
absorbed by country effects. Other slopes are common across countries; fixed
effects do not remove all time-varying confounding or establish causality.

The dummy-variable implementation follows Lecture 05 (slides 11–21) and handles
the slightly unbalanced complete-case sample. The static FE interpretation
retains the strict-exogeneity condition in slide 17; lagging regressors alone
does not establish it. Primary inference clusters by country, uses the
small-sample covariance correction, and refers tests to \(t(7)\). It allows
within-country dependence but assumes independence across countries; eight
clusters remain a material limitation (Lecture 05, slides 33–38 and 49–51).

### Working assumptions: monthly panel models

| Assumption | Defensibility and remaining risk |
|:--|:--|
| **Additive fixed effects** | Country effects capture time-invariant national differences and calendar-month effects absorb shocks common to countries in an additive manner. They cannot remove unobserved country-specific shocks that change over time. |
| **Strict exogeneity** | Standard static fixed-effects consistency requires the idiosyncratic error to be conditionally mean-independent of the regressors over the estimation window. A one-reference-month lag does **not** establish this: monetary policy, exchange rates and stock returns can respond to shared economic conditions. CPI reference dates do not equal historical publication dates. All slopes are therefore interpreted as conditional associations. |
| **Slope restrictions** | M1–M3 impose common macro slopes. M4 permits only the NEER slope to differ with euro-area membership; other group-specific macro effects remain omitted. Country classification is fixed within the sample, so its level effect is absorbed by country fixed effects. |
| **Country-clustered covariance** | The covariance estimator permits arbitrary within-country error dependence but assumes independence across countries. International shocks can violate that restriction; only eight country clusters limit reliability of asymptotic corrections and t(7) inference. The Driscoll–Kraay comparisons are sensitivity analyses, not proof that all inference assumptions hold. |
| **Comparable dependent variables** | All non-euro benchmarks are total-return indices, while three of four euro-area benchmarks are price indices. Unequal dividend treatment may introduce systematic measurement differences; fixed effects cannot remove time-varying dividend-return differences. |

The four model estimates are reported transparently, with their identification, measurement and covariance limitations. No causal effect of macroeconomic changes or euro-area membership is inferred.

In [ ]:
BASE_FORMULA = (
    "equity_return_pct ~ inflation_lag_pct "
    "+ policy_lag_pct + neer_lag_appreciation_pct"
)
COUNTRY_FORMULA = BASE_FORMULA + " + C(country)"
TWFE_FORMULA = COUNTRY_FORMULA + " + C(month_id)"
FORMULAS = {
    "M1 Pooled": BASE_FORMULA,
    "M2 Country FE": COUNTRY_FORMULA,
    "M3 Two-way FE": TWFE_FORMULA,
    "M4 NEER interaction": TWFE_FORMULA + " + neer_lag_appreciation_pct:euro",
}

In [ ]:
def country_cluster_fit(formula, data):
    """Estimate OLS with country-clustered, small-sample-corrected
    inference.
    """
    return smf.ols(formula, data=data).fit(
        cov_type="cluster",
        use_t=True,
        cov_kwds={
            "groups": data["country"],
            "use_correction": True,
            "df_correction": True,
        },
    )


def coefficient_table(fitted_models, selected_terms):
    """Collect selected coefficients and clustered inference without
    refitting models.
    """
    rows = []
    for name, fitted in fitted_models.items():
        for term in selected_terms:
            if term in fitted.params:
                rows.append(
                    {
                        "model": name,
                        "term": term,
                        "coef": fitted.params[term],
                        "country_cluster_SE": fitted.bse[term],
                        "p_value": fitted.pvalues[term],
                        "N": int(fitted.nobs),
                        "R_squared": fitted.rsquared,
                        "adjusted_R_squared": fitted.rsquared_adj,
                    }
                )
    return pd.DataFrame(rows)

In [ ]:
# All four specifications are fitted on the same complete-case observations.
models = {
    name: country_cluster_fit(formula, monthly_sample)
    for name, formula in FORMULAS.items()
}
terms = [*macro_terms, "neer_lag_appreciation_pct:euro"]

focal_model = models["M4 NEER interaction"]

In [ ]:
regression_table = coefficient_table(models, terms)
show_table(
    regression_table[
        ["model", "term", "coef", "country_cluster_SE", "p_value"]
    ]
    .replace({"term": TERM_LABELS})
    .rename(
        columns={
            "model": "Model",
            "term": "Regressor",
            "coef": "Coefficient",
            "country_cluster_SE": "Cluster SE",
            "p_value": "p-value",
        }
    ),
    "Four main monthly models: country-clustered inference",
    formats={"p-value": "{:.3g}"},
)

In [ ]:
show_table(
    regression_table[["model", "N", "R_squared", "adjusted_R_squared"]]
    .drop_duplicates()
    .rename(
        columns={
            "model": "Model",
            "R_squared": "R-squared",
            "adjusted_R_squared": "Adjusted R-squared",
        }
    ),
    "Main-model sample sizes and in-sample fit",
)

### Explanatory power after fixed effects

Overall R-squared includes country and month effects. A fixed-effects-only
reference, fitted on the same sample, provides the denominator for
\(1-SSR_{\mathrm{full}}/SSR_{\mathrm{FE}}\): the fraction of remaining variation
explained by the added macro terms, including M4's interaction. This is an
in-sample fit calculation, not a fifth main specification or a forecasting test.

In [ ]:
# The fixed-effects-only regression is a fit reference, not a fifth main model.
fe_reference = smf.ols(
    "equity_return_pct ~ C(country) + C(month_id)", data=monthly_sample
).fit()

fit_rows = [
    {
        "model": "Fixed effects only (reference)",
        "N": int(fe_reference.nobs),
        "R_squared": fe_reference.rsquared,
        "partial_R_squared_given_FE": np.nan,
    }
]

for name in ["M3 Two-way FE", "M4 NEER interaction"]:
    fitted = models[name]
    fit_rows.append(
        {
            "model": name,
            "N": int(fitted.nobs),
            "R_squared": fitted.rsquared,
            "partial_R_squared_given_FE": 1 - fitted.ssr / fe_reference.ssr,
        }
    )

fit_table = pd.DataFrame(fit_rows)

In [ ]:
show_table(
    fit_table.rename(
        columns={
            "model": "Model",
            "R_squared": "Overall R-squared",
            "partial_R_squared_given_FE": "Partial R-squared after FE",
        }
    ),
    "Additional explanatory power after country and month effects",
    formats={
        "Overall R-squared": "{:.6f}",
        "Partial R-squared after FE": "{:.6f}",
    },
)

The high overall fit largely reflects the fixed effects. The partial
R-squared separately identifies the incremental explanatory power of the
macroeconomic terms; overall fit should not be attributed to those terms alone.

## 8. Euro-area NEER comparison and inference sensitivity

In M4, the non-euro slope is \(\beta_q\), the euro slope is \(\beta_q+\delta\),
and the difference is \(\delta\). The comparison therefore tests

\[
H_0:\delta=0,
\]

alongside each group's slope against zero. A significant slope in one group
and an insignificant slope in another would not itself establish a difference
(Lecture 05, slides 70–71). Coefficients relate percentage-point equity returns
to a one-unit change in prior-month NEER appreciation measured as log change
multiplied by 100.

Country-clustered inference is compared with the retained Driscoll–Kraay
sensitivity estimates at 3, 6 and 12 monthly lags. The latter use Bartlett
weights, the \(n/(n-k)\) covariance correction and asymptotic normal tests and
95% intervals; the country-clustered tests use \(t(7)\). All bandwidths are
reported, so none is selected by significance. Driscoll–Kraay is an externally
referenced extension, not a method presented in Lecture 05; it relies on
long-time-series, weak-dependence conditions and does not address endogeneity.

In [ ]:
# Driscoll-Kraay uses calendar month identifiers shared across countries.
month_periods = pd.PeriodIndex(monthly_sample["month"], freq="M")
time_index = month_periods.asi8 - month_periods.asi8.min()

inference_variants = {"Country cluster (t, df=7)": focal_model}
for bandwidth in (3, 6, 12):
    inference_variants[f"DK ({bandwidth} monthly lags, normal)"] = (
        focal_model.get_robustcov_results(
            cov_type="hac-groupsum",
            time=time_index,
            maxlags=bandwidth,
            use_correction="hac",
            df_correction=False,
            use_t=False,
        )
    )

In [ ]:
neer_term = "neer_lag_appreciation_pct"
slope_hypotheses = {
    "Non-euro": f"{neer_term} = 0",
    "Euro": f"{neer_term} + {neer_term}:euro = 0",
    "Euro minus non-euro": f"{neer_term}:euro = 0",
}

slope_rows = []
for inference_name, fitted in inference_variants.items():
    for group, hypothesis in slope_hypotheses.items():
        test = fitted.t_test(hypothesis)
        lower, upper = test.conf_int(alpha=0.05).ravel()
        slope_rows.append(
            {
                "inference": inference_name,
                "group": group,
                "estimate": test.effect.item(),
                "std_error": test.sd.item(),
                "p_vs_zero": float(test.pvalue),
                "ci_lower": lower,
                "ci_upper": upper,
            }
        )

neer_effects = pd.DataFrame(slope_rows)

In [ ]:
show_table(
    neer_effects.set_index(["inference", "group"]).rename(
        columns={
            "estimate": "Slope",
            "std_error": "SE",
            "p_vs_zero": "p-value",
            "ci_lower": "95% CI lower",
            "ci_upper": "95% CI upper",
        }
    ),
    "NEER slopes: country-clustered and Driscoll-Kraay inference",
    formats={"p-value": "{:.3g}"},
    keep_index=True,
)

The estimated euro-minus-non-euro slope difference is negative, but neither
country-clustered inference nor the reported Driscoll–Kraay comparisons reject
equal slopes at 5%. This is insufficient evidence of a difference, not evidence
that the relationships are identical. Panel uncertainty warrants caution with
only eight countries.

## 9. Global-recession-year sensitivity

M4 is re-estimated after excluding every month in 2009 and 2020. The World
Bank (2026, Box 1.1, p. 8) identifies these as global recession years using an
annual decline in global real GDP per capita. This definition does not identify
country-specific recession months or remove all national downturns.

The specification and country-clustered inference are held fixed to isolate the
sample change. Lags are constructed before exclusion, so January 2010 still uses
December 2009 information. A common recession indicator is not added to the
complete calendar-month effects, which would absorb it. This is a sensitivity
check rather than a fifth main model.

In [ ]:
# Create recession sensitivity subsample AFTER calculating all monthly lags.
# Otherwise filtering first could misalign the explanatory variables.
sensitivity_data = monthly_sample.loc[
    ~monthly_sample["year"].isin(RECESSION_YEARS)
].copy()

robust_model = country_cluster_fit(
    FORMULAS["M4 NEER interaction"], sensitivity_data
)

robustness_models = {
    "Main two-way FE + NEER interaction": focal_model,
    f"Exclude global recession years {RECESSION_YEARS}": robust_model,
}
robustness_table = (
    coefficient_table(robustness_models, terms)
    .rename(columns={"model": "specification", "coef": "coefficient"})
    .drop(columns="adjusted_R_squared")
)

In [ ]:
show_table(
    robustness_table.replace(
        {
            "term": TERM_LABELS,
            "specification": {
                "Main two-way FE + NEER interaction": "Full sample",
                (
                    "Exclude global recession years " f"{RECESSION_YEARS}"
                ): "Exclude 2009/2020",
            },
        }
    ).rename(
        columns={
            "specification": "Sample",
            "term": "Regressor",
            "coefficient": "Coefficient",
            "country_cluster_SE": "Cluster SE",
            "p_value": "p-value",
            "R_squared": "R-squared",
        }
    ),
    "M4 global-recession-year sensitivity: country-clustered inference",
    formats={"p-value": "{:.3g}"},
)

print("Regression rows excluded:", len(monthly_sample) - len(sensitivity_data))

Excluding the two global recession years preserves the negative sign of the
estimated group difference without making it statistically significant. The
annual exclusion and eight-country inference limitations remain relevant to
its interpretation.

## 10. Dividend-convention sensitivity: harmonised total-return benchmarks

**Question.** Does the estimated euro-minus-non-euro NEER slope difference in M4
change when the three price-return equity benchmarks are replaced by their
gross total-return counterparts? The original full-period result is retained;
this is a **supplementary sensitivity analysis**, not a fifth main model.

**Design and identifying assumptions.** We use two *identical* balanced
country-month samples, July 2014–September 2026 (147 months × 8 countries).
The baseline M4 uses the original indices; the alternative changes only
Austria, Finland and Italy to `ATXTR`, `OMXH25GI` and `TFTSEMIB`. Germany,
Brazil, Israel, Norway and Poland retain their original total-return
benchmarks. All three new Bloomberg exports have monthly `PX_LAST` levels in
EUR, and their source formulas request the terminal DPDF settings
(`UseDPDF=Y`). The DPDF screen inspected on 16 November showed both
historical cash-dividend adjustment options unchecked (Section 2); our
analysis applies no further dividend adjustment. Simple monthly returns
are calculated from **consecutive** monthly levels before sample selection.

This design controls for the sample-window change when assessing sensitivity
to index choice, but does **not** prove that any difference is caused solely
by dividends: provider conventions, rebalancing, data-vendor settings and
rounding may also matter. The models remain associational; fixed-effects
exogeneity, eight-country-cluster inference and possible cross-country
correlation limitations remain. A direct regression of the return-series
difference is an additional descriptive check, not an equivalence test.


In [ ]:
# Read supplementary source-level Bloomberg PX_LAST data from the SAME
# submission workbook. The original four models above are not refitted.
TR_TICKERS = {
    "Austria": "ATXTR Index",
    "Finland": "OMXH25GI Index",
    "Italy": "TFTSEMIB Index",
}
TR_START = pd.Period("2014-07", freq="M")
TR_END = MONTH_END
TR_MONTHS = pd.period_range(TR_START, TR_END, freq="M")

tr_levels = {}
tr_returns = {}
tr_coverage_rows = []
for country, ticker in TR_TICKERS.items():
    sheet_name = f"TR_{country}"
    levels = read_source(DATA_FILE, sheet_name)
    assert list(levels.columns) == ["Date", "PX_LAST"]
    assert (levels["PX_LAST"] > 0).all(), f"Non-positive TR level: {country}"
    levels["month"] = levels["Date"].dt.to_period("M")
    assert levels["month"].is_unique
    assert levels["month"].is_monotonic_increasing
    assert levels["month"].tolist() == list(pd.period_range(
        levels["month"].min(), levels["month"].max(), freq="M"
    )), f"Missing months in {country} TR levels"
    assert levels["month"].min() <= pd.Period("2014-06", freq="M")
    assert levels["month"].max() >= TR_END
    # The source sheet's first-row caption retains the exact Bloomberg ticker.
    source_caption = pd.read_excel(
        DATA_FILE, sheet_name=sheet_name, header=None, nrows=1
    ).iat[0, 0]
    assert ticker in source_caption, f"Unexpected Bloomberg ticker: {country}"
    levels["return_pct"] = 100 * (
        levels["PX_LAST"] / levels["PX_LAST"].shift(1) - 1
    )
    tr_levels[country] = levels
    tr_returns[country] = levels.set_index("month")["return_pct"]
    assert tr_returns[country].reindex(TR_MONTHS).notna().all(), (
        f"Incomplete common-period TR returns: {country}"
    )
    tr_coverage_rows.append({
        "country": country,
        "ticker": ticker,
        "first_level_month": str(levels["month"].min()),
        "last_level_month": str(levels["month"].max()),
        "monthly_levels": len(levels),
        "missing_common_returns": int(tr_returns[country].reindex(TR_MONTHS).isna().sum()),
    })
tr_coverage = pd.DataFrame(tr_coverage_rows)
show_table(tr_coverage, "New Bloomberg total-return series: source coverage")

# Exact same country-month rows and macro regressors for both M4 fits.
tr_common = monthly_sample.loc[
    monthly_sample["month"].between(TR_START, TR_END)
].copy()
expected_keys = pd.MultiIndex.from_product(
    [COUNTRIES, TR_MONTHS], names=["country", "month"]
)
actual_keys = pd.MultiIndex.from_frame(tr_common[["country", "month"]])
assert len(tr_common) == len(expected_keys) == 8 * 147
assert actual_keys.is_unique and actual_keys.difference(expected_keys).empty
assert expected_keys.difference(actual_keys).empty
tr_common["equity_return_harmonised_pct"] = tr_common["equity_return_pct"]
for country, series in tr_returns.items():
    mask = tr_common["country"].eq(country)
    tr_common.loc[mask, "equity_return_harmonised_pct"] = (
        tr_common.loc[mask, "month"].map(series).to_numpy()
    )
assert tr_common["equity_return_harmonised_pct"].notna().all()
assert np.isfinite(tr_common[["equity_return_pct", "equity_return_harmonised_pct", *macro_terms]]).all().all()
print(f"Validated matched comparison: {len(tr_common):,} country-months, "
      f"{TR_START} through {TR_END}; same regressors and country-month fixed effects.")


In [ ]:
# Country-clustered M4 is estimated on precisely the same observations.
tr_original_fit = country_cluster_fit(
    FORMULAS["M4 NEER interaction"], tr_common
)
tr_harmonised_fit = country_cluster_fit(
    FORMULAS["M4 NEER interaction"],
    tr_common.assign(equity_return_pct=tr_common["equity_return_harmonised_pct"]),
)
# The paired return difference isolates the change in the dependent variable,
# holding the sample, regressors, fixed effects and covariance rule fixed.
tr_paired_fit = country_cluster_fit(
    FORMULAS["M4 NEER interaction"],
    tr_common.assign(
        equity_return_pct=(
            tr_common["equity_return_harmonised_pct"] - tr_common["equity_return_pct"]
        )
    ),
)
tr_term = "neer_lag_appreciation_pct:euro"
tr_model_map = {
    "Original M4, full period": focal_model,
    "Original M4, common period": tr_original_fit,
    "Gross TR M4, common period": tr_harmonised_fit,
    "Paired TR-minus-original difference": tr_paired_fit,
}
tr_model_rows = []
for label, fitted in tr_model_map.items():
    ci = fitted.conf_int(alpha=0.05).loc[tr_term]
    tr_model_rows.append({
        "model": label,
        "N": int(fitted.nobs),
        "interaction": fitted.params[tr_term],
        "cluster_SE": fitted.bse[tr_term],
        "p_value": fitted.pvalues[tr_term],
        "ci_lower": ci.iloc[0],
        "ci_upper": ci.iloc[1],
        "R_squared": fitted.rsquared,
    })
tr_model_comparison = pd.DataFrame(tr_model_rows)
assert np.isclose(
    tr_harmonised_fit.params[tr_term] - tr_original_fit.params[tr_term],
    tr_paired_fit.params[tr_term], atol=1e-10
)
show_table(
    tr_model_comparison,
    "NEER × euro sensitivity: same-window original vs gross total-return M4",
)

# Compare observed return differences and flag any non-dividend-like anomalies.
tr_difference_rows = []
for country in TR_TICKERS:
    subset = tr_common.loc[tr_common["country"].eq(country)]
    difference = subset["equity_return_harmonised_pct"] - subset["equity_return_pct"]
    tr_difference_rows.append({
        "country": country,
        "N": len(subset),
        "mean_difference_pp": difference.mean(),
        "median_difference_pp": difference.median(),
        "min_difference_pp": difference.min(),
        "max_difference_pp": difference.max(),
        "months_below_minus_0_01_pp": int((difference < -0.01).sum()),
        "return_correlation": subset[
            ["equity_return_pct", "equity_return_harmonised_pct"]
        ].corr().iloc[0, 1],
    })
tr_return_differences = pd.DataFrame(tr_difference_rows)
show_table(tr_return_differences, "Monthly replacement-index return differences and QA")


In [ ]:
# Inference robustness: compare Driscoll-Kraay SEs using the same monthly
# time index, and exclude the first two months as an early-history check.
tr_time_periods = pd.PeriodIndex(tr_common["month"], freq="M")
tr_time = tr_time_periods.asi8 - tr_time_periods.asi8.min()
tr_dk_rows = []
for name, fitted in {
    "Original common period": tr_original_fit,
    "Gross TR common period": tr_harmonised_fit,
}.items():
    for bandwidth in (3, 6, 12):
        dk = fitted.get_robustcov_results(
            cov_type="hac-groupsum", time=tr_time, maxlags=bandwidth,
            use_correction="hac", df_correction=False, use_t=False,
        )
        term_idx = fitted.model.exog_names.index(tr_term)
        tr_dk_rows.append({
            "model": name,
            "DK_monthly_lags": bandwidth,
            "interaction": fitted.params[tr_term],
            "DK_SE": dk.bse[term_idx],
            "p_value": dk.pvalues[term_idx],
        })
tr_dk_sensitivity = pd.DataFrame(tr_dk_rows)
show_table(tr_dk_sensitivity, "Common-period DK covariance sensitivity")

# Finland's July/August 2014 return differences approximately offset;
# removing both calendar months from BOTH specifications tests dependence on
# these earliest months without changing the balanced country design.
tr_later = tr_common.loc[
    tr_common["month"] >= pd.Period("2014-09", freq="M")
].copy()
tr_later_original = country_cluster_fit(
    FORMULAS["M4 NEER interaction"], tr_later
)
tr_later_harmonised = country_cluster_fit(
    FORMULAS["M4 NEER interaction"],
    tr_later.assign(equity_return_pct=tr_later["equity_return_harmonised_pct"]),
)
assert len(tr_later) == 8 * 145
print(
    "Excluding July–August 2014 (same 1,160 observations): "
    f"original NEER×euro = {tr_later_original.params[tr_term]:.4f}; "
    f"gross TR = {tr_later_harmonised.params[tr_term]:.4f}."
)


**Interpretation and limits.** The full-sample M4 interaction is about
−0.4825 (country-clustered p ≈ 0.176). On the *same* July 2014–September 2026
sample, the original indices give −0.2374 (p ≈ 0.523) and the gross
TR replacements give −0.2450 (p ≈ 0.449). Their difference is −0.0076;
the paired return-difference regression gives p ≈ 0.882. Thus the group
interaction estimate is **not materially sensitive to the replacement indices
within this period**. It remains statistically inconclusive, not proven zero.

The full-sample to shorter-period change is a *sample-period effect*, not
an effect of dividend harmonisation. The new TR series have higher average
monthly returns than the original price indices (as expected when dividends
are included), but the Finnish July–August 2014 deviations warrant a
qualification: this is a matched **index-choice sensitivity**, not proof that
only dividend treatment differs. The 2014-09 onward check and DK standard-error
comparisons do not reverse the qualitative conclusion. No causal NEER
interpretation or definitive validation of eight-cluster inference is claimed.


## 11. Supplementary inference and influential-country checks (M4 only)

**Assessment remit.** Section B asks for up to four *main* panel specifications and critical discussion of inference and limitations. These are diagnostics and sensitivity refits of the pre-specified **M4**, not additional main specifications. The principal M1–M4 estimates, recession exclusion and harmonised-return analysis above are unchanged.

**Leave-one-country-out (LOCO).** Refit exactly the M4 formula eight times, removing one whole national panel at a time. This examines concentration of the NEER×euro interaction in a single country. Excluding a country changes the country composition (including the euro/non-euro balance) and is not a causal test; the six remaining reference degrees of freedom reflect seven country clusters.

**Cross-country residual association.** Report pairwise correlations of M4 residuals on common calendar months. This is *descriptive*, **not** a Pesaran-CD significance test: calendar-month fixed effects impose a within-month zero-sum constraint that can mechanically induce negative residual correlations, and cross-country residual dependence is not uniquely identified by this matrix.

**Bias-reduced cluster-robust sensitivity (CR2).** Compute the CR2 variance for the M4 NEER×euro coefficient with the full OLS design including fixed effects. Each cluster adjustment uses the Moore–Penrose inverse square root of \(I-H_{gg}\) (a country dummy makes one direction singular). A Bell–McCaffrey/Satterthwaite approximation supplies *coefficient-specific* degrees of freedom under an independent, homoskedastic working model. CR2 does not resolve genuine dependence *between* clusters or establish strict exogeneity, and with eight clusters its reference distribution remains fragile. Country-clustered t(7) and Driscoll–Kraay results above remain the primary comparison.

In [ ]:
# Supplementary leave-one-country-out M4 refits; do not alter M1–M4.
looc_rows = []
interaction = "neer_lag_appreciation_pct:euro"
for omitted in COUNTRIES:
    restricted = monthly_sample.loc[monthly_sample["country"] != omitted].copy()
    fitted = country_cluster_fit(FORMULAS["M4 NEER interaction"], restricted)
    assert restricted["country"].nunique() == 7
    looc_rows.append({
        "omitted_country": omitted,
        "N": int(fitted.nobs),
        "remaining_euro_countries": int(restricted.loc[restricted.euro.eq(1), "country"].nunique()),
        "remaining_non_euro_countries": int(restricted.loc[restricted.euro.eq(0), "country"].nunique()),
        "NEER_euro_coefficient": float(fitted.params[interaction]),
        "cluster_SE": float(fitted.bse[interaction]),
        "p_value_t6": float(fitted.pvalues[interaction]),
    })
looc_table = pd.DataFrame(looc_rows)
show_table(
    looc_table.rename(columns={
        "omitted_country": "Country omitted",
        "N": "N",
        "remaining_euro_countries": "Euro countries left",
        "remaining_non_euro_countries": "Non-euro countries left",
        "NEER_euro_coefficient": "NEER × euro estimate",
        "cluster_SE": "Cluster SE",
        "p_value_t6": "p-value, t(6)",
    }),
    "M4 leave-one-country-out sensitivity (7 clusters)",
)
print("Full-sample NEER×euro coefficient:", round(focal_model.params[interaction], 5))
print("LOCO coefficient range:", round(looc_table.NEER_euro_coefficient.min(), 5),
      "to", round(looc_table.NEER_euro_coefficient.max(), 5))


In [ ]:
# M4 residual correlation matrix: descriptive only, since calendar-month
# fixed effects create mechanical within-month constraints.
residual_frame = monthly_sample[["country", "month"]].copy()
residual_frame["residual"] = np.asarray(focal_model.resid)
residual_wide = residual_frame.pivot(index="month", columns="country", values="residual")
residual_corr = residual_wide.corr().reindex(index=COUNTRIES, columns=COUNTRIES)
upper_idx = np.triu_indices(len(COUNTRIES), k=1)
pairwise_r = residual_corr.to_numpy()[upper_idx]
residual_summary = pd.DataFrame([{
    "country_pairs": len(pairwise_r),
    "mean_pairwise_correlation": np.mean(pairwise_r),
    "mean_absolute_pairwise_correlation": np.mean(np.abs(pairwise_r)),
    "maximum_absolute_pairwise_correlation": np.max(np.abs(pairwise_r)),
    "min_pairwise_overlap_months": int((residual_wide.notna().T.astype(int) @ residual_wide.notna().astype(int)).to_numpy()[upper_idx].min()),
}])
show_table(residual_corr, "M4 residual correlations by country (descriptive)", keep_index=True)
show_table(residual_summary, "Descriptive residual-correlation summary")


In [ ]:
# CR2 for the focal interaction, with coefficient-specific Satterthwaite df.
# The full OLS design includes every country/month indicator; its restricted
# within-country leverage matrices have a null direction for each country FE.
# Pseudoinverse adjustment is appropriate for that structurally singular direction.
from scipy import linalg as sla
from scipy.stats import t as student_t

X = np.asarray(focal_model.model.exog, dtype=float)
e = np.asarray(focal_model.resid, dtype=float)
y = np.asarray(focal_model.model.endog, dtype=float)
assert X.shape[0] == len(monthly_sample) == len(e)
XtX_inv = sla.pinvh(X.T @ X, rtol=1e-11)
term_pos = focal_model.model.exog_names.index(interaction)
cluster_labels = monthly_sample["country"].to_numpy()
cluster_list = list(COUNTRIES)
cluster_contributions = []
Z = np.zeros((len(cluster_list), len(e)))
cr2_eigen_report = []
for gi, country in enumerate(cluster_list):
    ids = np.flatnonzero(cluster_labels == country)
    Xg = X[ids]
    Hgg = Xg @ XtX_inv @ Xg.T
    unadjusted = np.eye(len(ids)) - (Hgg + Hgg.T) / 2
    eigenvalues, U = sla.eigh(unadjusted, check_finite=True)
    # CR2's homoskedastic working model makes a cluster-dummy direction singular.
    tol = 1e-8
    assert eigenvalues.min() > -tol, "Unexpected non-PSD leverage complement"
    inv_sqrt = np.where(eigenvalues > tol, 1 / np.sqrt(np.maximum(eigenvalues, tol)), 0)
    A = (U * inv_sqrt) @ U.T
    aj = Xg @ XtX_inv[:, term_pos]
    contribution = float(aj @ (A @ e[ids]))
    cluster_contributions.append(contribution)
    Z[gi, ids] = A @ aj
    cr2_eigen_report.append({"country": country, "structural_null_directions": int((eigenvalues <= tol).sum()),
                             "smallest_positive_eigenvalue": float(eigenvalues[eigenvalues > tol].min())})
cr2_variance = float(np.sum(np.square(cluster_contributions)))
cr2_se = np.sqrt(cr2_variance)
# Under iid homoskedastic working errors, projection residuals are (I-H)epsilon.
# Satterthwaite df = [tr(C)]^2 / tr(C^2), where C=QQ' and rows of Q are
# coefficient-specific adjusted cluster linear forms applied to (I-H).
Q = Z - ((Z @ X) @ XtX_inv) @ X.T
C = Q @ Q.T
cr2_df = float(np.trace(C) ** 2 / np.square(C).sum())
assert 0 < cr2_df <= len(COUNTRIES) + 1e-6
cr2_t = float(focal_model.params[interaction] / cr2_se)
cr2_p = float(2 * student_t.sf(abs(cr2_t), df=cr2_df))
cr2_crit = student_t.ppf(0.975, df=cr2_df)
cr2_table = pd.DataFrame([
    {"inference": "Country cluster CR1, t(7)", "N": int(focal_model.nobs),
     "estimate": float(focal_model.params[interaction]),
     "SE": float(focal_model.bse[interaction]), "reference_df": 7.,
     "p_value": float(focal_model.pvalues[interaction]),
     "CI_low": float(focal_model.conf_int().loc[interaction].iloc[0]),
     "CI_high": float(focal_model.conf_int().loc[interaction].iloc[1])},
    {"inference": "CR2, Satterthwaite t", "N": int(focal_model.nobs),
     "estimate": float(focal_model.params[interaction]),
     "SE": cr2_se, "reference_df": cr2_df, "p_value": cr2_p,
     "CI_low": float(focal_model.params[interaction] - cr2_crit * cr2_se),
     "CI_high": float(focal_model.params[interaction] + cr2_crit * cr2_se)},
])
show_table(
    cr2_table.rename(columns={
        "inference": "Inference",
        "N": "N",
        "estimate": "Estimate",
        "SE": "SE",
        "reference_df": "Reference df",
        "p_value": "p-value",
        "CI_low": "95% CI lower",
        "CI_high": "95% CI upper",
    }),
    "NEER×euro inference: country-clustered CR1 vs CR2",
)
assert all(row["structural_null_directions"] >= 1 for row in cr2_eigen_report)
print("CR2 null directions by cluster:", [(r["country"], r["structural_null_directions"]) for r in cr2_eigen_report])


**Result and interpretation.** Leave-one-country-out M4 interaction estimates range from **−0.712** (excluding Finland, country-clustered t(6) p ≈ **0.063**) to **−0.261** (excluding Italy, p ≈ **0.369**); all eight estimates retain the negative sign and none meets 5% significance. This indicates *magnitude sensitivity* to panel composition, not proof of an invariant euro/non-euro effect. The unmodified primary M4 gives δ = **−0.4825** (CR1 SE **0.3201**, t(7) p **0.1755**). Bias-reduced CR2 gives SE **0.3282**, Satterthwaite degrees of freedom ≈ **5.09**, and p ≈ **0.2005**, also inconclusive. These are coefficient-specific finite-sample sensitivity calculations under an independence/homoskedasticity *working model*, not a guarantee of valid inference when countries are dependent. The descriptive residual-correlation matrix has mean absolute off-diagonal correlation **0.181** and maximum **0.445**, but *cannot* identify residual cross-country independence because calendar-month fixed effects themselves impose cross-sectional constraints. Strict exogeneity and real-time CPI availability remain unverified. These checks extend **M4 diagnostics**, not the coursework's maximum of four main specifications (Pustejovsky and Tipton, 2018).

In [ ]:
# Export only the supplementary diagnostic tables. These files do not
# modify the submission Excel data, original results or four main models.
looc_table.to_csv(OUTPUT_DIR / "supplement_m4_leave_one_country_out.csv", index=False)
residual_corr.to_csv(OUTPUT_DIR / "supplement_m4_residual_correlation_matrix.csv")
residual_summary.to_csv(OUTPUT_DIR / "supplement_m4_residual_correlation_summary.csv", index=False)
cr2_table.to_csv(OUTPUT_DIR / "supplement_m4_cr2_inference.csv", index=False)


## 12. Validation and exports

The final assertions check finite analysis values and common main-model sample
sizes; date, input and panel validations also run at their relevant stages.
Tables are exported at full precision, while the two figures are saved in their
plotting cells before being closed. The PDF report should quote the estimates
and uncertainty measures from these executed outputs.

In [ ]:
# Fail on malformed final data or mismatched regression samples.
assert (
    np.isfinite(daily_sample[["siemens_return_pct", "dax_return_pct"]])
    .all()
    .all()
)
assert np.isfinite(monthly_sample[regression_columns]).all().all()
assert all(int(model.nobs) == len(monthly_sample) for model in models.values())
assert np.isfinite(focal_model.params).all()

print(
    f"Validation complete: {len(daily_sample):,} daily-return pairs and "
    f"{len(monthly_sample):,} monthly regression rows."
)
print(f"Reproducible tables and figures: {OUTPUT_DIR}/")
print("Bloomberg source verification and new index exports are documented in Section 13.")

In [ ]:
package_versions.to_csv(OUTPUT_DIR / "package_versions.csv")
coverage.to_csv(OUTPUT_DIR / "source_coverage.csv", index=False)
daily_sample.to_csv(OUTPUT_DIR / "daily_returns.csv", index=False)
daily_coefficients.to_csv(OUTPUT_DIR / "part1_baseline_coefficients.csv")
pd.DataFrame(
    {
        "BP_lm_p": [bp_lm_p],
        "BP_f_p": [bp_f_p],
        "LjungBox5_p": [ljung_box.loc[5, "lb_pvalue"]],
        "LjungBox20_p": [ljung_box.loc[20, "lb_pvalue"]],
    }
).to_csv(OUTPUT_DIR / "part1_diagnostics.csv", index=False)
period_comparison.to_csv(OUTPUT_DIR / "part1_beta_periods.csv", index=False)
stability_tests.to_csv(OUTPUT_DIR / "part1_stability_tests.csv", index=False)
monthly_sample.to_csv(OUTPUT_DIR / "monthly_analysis_panel.csv", index=False)
country_summary.to_csv(OUTPUT_DIR / "monthly_country_summary.csv")
regression_table.to_csv(
    OUTPUT_DIR / "part2_four_model_coefficients.csv", index=False
)
fit_table.to_csv(OUTPUT_DIR / "part2_fit_after_fixed_effects.csv", index=False)
neer_effects.to_csv(
    OUTPUT_DIR / "part2_neer_group_comparison.csv", index=False
)
robustness_table.to_csv(
    OUTPUT_DIR / "part2_recession_robustness.csv", index=False
)
# Supplementary harmonised-index QA and M4 comparison outputs.
tr_coverage.to_csv(OUTPUT_DIR / "tr_source_coverage.csv", index=False)
tr_return_differences.to_csv(OUTPUT_DIR / "tr_return_differences.csv", index=False)
tr_model_comparison.to_csv(OUTPUT_DIR / "tr_m4_sensitivity_comparison.csv", index=False)
tr_dk_sensitivity.to_csv(OUTPUT_DIR / "tr_dk_inference_sensitivity.csv", index=False)


## 13. References and source-verification record

### Lecture-based econometric methods

The printed lecture-slide numbers below are not always identical to the PDF
page counts, because some PDFs contain successive builds.

- Calef, A. (2026). *MSIN0269 Lecture 02: Classical linear regression model I*.
  UCL School of Management. Slides 21, 37–38: model assumptions and errors.
- Calef, A. (2026). *MSIN0269 Lecture 03: Classical linear regression model II*.
  UCL School of Management. Slides 26–32, 35 and 41–43: parameter restrictions
  and model diagnostics.
- Calef, A. (2026). *MSIN0269 Topic 04: Time Series*.
  UCL School of Management. Slides 5, 7 and 10: returns, stationarity and Ljung–Box.
- Calef, A. (2026). *MSIN0269 Topic 05: Panel Data*.
  UCL School of Management. Slides 11–21, 33–51 and 70–71: fixed effects,
  covariance estimation, small-cluster uncertainty and slope interactions.

### Supplementary methods and data sources

- Pustejovsky, J.E. and Tipton, E. (2018). 'Small-sample methods for cluster-robust variance estimation and hypothesis testing in fixed effects models', *Journal of Business & Economic Statistics*, 36(4), 672–683. https://doi.org/10.1080/07350015.2016.1247004.
- Cameron, A.C. and Miller, D.L. (2015). 'A Practitioner's Guide to Cluster-Robust
  Inference', *Journal of Human Resources*, 50(2), 317–372.
- Newey, W.K. and West, K.D. (1987). 'A Simple, Positive Semi-definite,
  Heteroskedasticity and Autocorrelation Consistent Covariance Matrix',
  *Econometrica*, 55(3), 703–708.
- Driscoll, J.C. and Kraay, A.C. (1998). 'Consistent Covariance Matrix
  Estimation with Spatially Dependent Panel Data', *Review of Economics and
  Statistics*, 80(4), 549–560.
- World Bank (2026). *Global Economic Prospects*, January, Box 1.1.
  [Global recession classification](https://thedocs.worldbank.org/en/doc/7ce50b5aa95bef66048680bba9926ec8-0050012026/related/GEP-Jan-2026-Box-1-1.pdf).
- Bank for International Settlements (2026).
  [Central bank policy rates documentation](https://www.bis.org/statistics/cbpol/cbpol_doc.pdf).
- Bloomberg L.P. (2026). Bloomberg Terminal historical equity and CPI data;
  original formula strings and tickers are preserved in the workbook. Original equity/CPI series extracted 12 November 2026; DES and Siemens HP screens
  inspected 16 November 2026. Supplementary gross total-return exports for
  `ATXTR`, `OMXH25GI` and `TFTSEMIB` extracted 16 November 2026.
- Deutsche Börse (2026). [DAX benchmark and methodology](https://www.deutsche-boerse.com/dbg-en/media/news-stories/explainers/spotlight/DAX-Benchmark-and-Barometer-for-the-German-Stock-Market-139948).
- Vienna Stock Exchange (2026). [Index calculation details: ATX and ATX TR](https://www.wienerborse.at/en/indices/index-calculation/calculation-details/).
- Nasdaq (2021). [OMX Helsinki 25 index versions](https://indexes.nasdaqomx.com/docs/Index_Versions_OMXH25.pdf).
- FTSE Russell (2026). [FTSE MIB Index Ground Rules](https://www.lseg.com/content/dam/ftse-russell/en_us/documents/ground-rules/ftse-mib-index-english-ground-rules.pdf).

### Verification outcome and remaining administration

- **Index definition checks:** eight national benchmarks classified on
  16 November 2026 (see Section 2). These are index methodology conventions,
  not a retrospective recalculation of dividends in each daily/monthly row.
- **Siemens outlier checks:** 17 March 2008 and 29 October 2008 returns matched
  Bloomberg HP displayed historical closing prices; economic causes were not
  separately authenticated.
- **Original Siemens/DAX export settings:** historical Bloomberg formulas
  explicitly requested normal/special dividend and capital-change adjustments
  with `UseDPDF=N`. Original extraction was on 12 November 2026
  (student-reported); DES/HP verification was completed on 16 November 2026.
- **Supplementary index export settings:** original `ATXTR`, `OMXH25GI` and
  `TFTSEMIB` Excel BDH formulas specify `UseDPDF=Y`. The DPDF screen observed
  on 16 November 2026 showed historical normal/abnormal cash-dividend
  adjustments **off**, historical pricing/volume adjustments **on**, and
  a distinct intraday cash-dividend option **on**. The index-choice analysis
  uses the exported levels as supplied, without another dividend adjustment.
  The observed screen and formulas are source-setting evidence, not a
  historical transaction-by-transaction reconstruction.
- **Administrative item before submission:** confirm permitted redistribution
  of Bloomberg-derived Excel data under the applicable UCL licence. No licence
  approval is claimed here.

The raw-return Siemens market model, publication-date caveat for CPI, eight-
cluster inference limitations and annual recession exclusions remain documented
in their respective sections. The PDF report, executed notebook and supplied
Excel workbook must agree numerically.


### Supplementary harmonised-return sensitivity (Section 10)

- Source workbook tabs: `TR_Austria`, `TR_Finland`, `TR_Italy` and
  `TR_Source_Register`; original files `Austria_ATXTR.xlsx`,
  `Finland_OMXH25GI.xlsx`, `Italy_TFTSEMIB_Index.xlsx`.
- All new index levels are monthly Bloomberg `PX_LAST` in EUR. Available
  histories are January 2005–September 2026 (Austria), June 2014–September
  2026 (Finland), August 2011–September 2026 (Italy). No missing monthly
  levels or duplicates were observed.
- Two M4 regressions share exactly 1,176 country-months, regressors,
  country/month effects and country-clustered inference. The full-period
  original M4 is unchanged. The supplementary analysis cannot identify
  dividends as the unique cause of any difference between benchmark series.
